# AI-Driven OSINT Threat Monitoring & Risk Assessment
**KTP Associate Technical Assessment — Prototype Notebook**

This notebook demonstrates a complete end-to-end pipeline:
1. Data collection from The Guardian Open Platform API
2. Text preprocessing (HTML stripping, cleaning)
3. Named Entity Recognition (spaCy)
4. Threat classification (keyword-based + zero-shot NLI)
5. Composite risk scoring
6. Visualisations & threat report generation

---
**Data source:** The Guardian Open Platform (`https://open-platform.theguardian.com/`)  
**Author:** Sina Ebrahimi  
**Date:** 2026-06-03

## 0 — Setup & Imports

Ensure the `src/` package is on the Python path and all dependencies are available.
Run `setup.ps1` first if you haven't already.

In [ ]:
import sys, pathlib

# Add the project root to sys.path so `src` is importable
ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
from IPython.display import display, HTML

# Project modules
from src.config import (
    THREAT_TAXONOMY, OUTPUTS, DATE_FROM, DATE_TO, THREAT_QUERIES
)
from src.collector import GuardianCollector
from src.preprocessor import preprocess_dataframe
from src.ner import enrich_dataframe
from src.classifier import classify_dataframe
from src.risk_scorer import score_dataframe
from src.reporter import generate_threat_report, export_report, print_summary, intelligence_briefs

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
})
PALETTE = ['#E63946', '#F4A261', '#2A9D8F', '#457B9D', '#6D2B82', '#264653']

print(f"Project root: {ROOT}")
print(f"Collection window: {DATE_FROM} → {DATE_TO}")
print(f"Number of threat queries: {len(THREAT_QUERIES)}")
print(f"Threat categories: {list(THREAT_TAXONOMY.keys())}")

## 1 — Data Collection

We query The Guardian Open Platform for each threat term in `THREAT_QUERIES`.
Articles are deduplicated by ID (the same article may surface for multiple queries).

If `data/raw/guardian_raw.json` already exists, reload it to avoid re-fetching the API.

In [ ]:
import json
from src.config import DATA_RAW

raw_path = DATA_RAW / 'guardian_raw.json'

if raw_path.exists():
    print(f"Loading cached articles from {raw_path} ...")
    df_raw = GuardianCollector.load_raw(raw_path)
else:
    print("Fetching articles from Guardian API (this may take 1-2 minutes) ...")
    collector = GuardianCollector()
    df_raw = collector.collect_threat_corpus()

print(f"\n✓ Dataset: {len(df_raw)} unique articles")
print(f"  Date range: {df_raw['published'].min().date()} → {df_raw['published'].max().date()}")
print(f"  Sections: {df_raw['section'].value_counts().head(5).to_dict()}")
df_raw.head(3)

In [ ]:
# Dataset overview
print("Column overview:")
display(df_raw.dtypes.to_frame('dtype').join(df_raw.isnull().mean().rename('null_frac')))

print(f"\nSection distribution (top 10):")
display(df_raw['section'].value_counts().head(10))

## 2 — Preprocessing

Steps:
- Strip HTML tags from `body_html` (Guardian returns full article bodies as HTML)
- Normalise whitespace and remove control characters
- Build a `search_text` field: `title × 3 + standfirst + body` (title is upweighted for classification)

In [ ]:
df_clean = preprocess_dataframe(df_raw)

print(f"\n✓ After preprocessing: {len(df_clean)} articles")
print(f"  Avg body length (chars): {df_clean['body_clean'].str.len().mean():.0f}")
print(f"  Avg word count: {df_clean['wordcount'].mean():.0f}")

# Show a before/after HTML stripping example
idx = df_clean['body_html'].str.len().idxmax()
print("\n--- HTML body snippet (raw) ---")
print(df_clean.loc[idx, 'body_html'][:400])
print("\n--- After stripping ---")
print(df_clean.loc[idx, 'body_clean'][:400])

## 3 — Named Entity Recognition

We use **spaCy `en_core_web_sm`** to extract:
- `ORG` — organisations (companies, agencies, governments)
- `GPE` / `LOC` — geopolitical entities (countries, cities)
- `PERSON` — named individuals
- `EVENT` — named events
- `NORP` — nationalities, religious or political groups

Entity density serves as a risk signal: a highly specific article (naming particular actors and locations) is more actionable threat intelligence than a vague one.

In [ ]:
df_ner = enrich_dataframe(df_clean)

print(f"\n✓ NER complete")
print(f"  Avg entity count per article: {df_ner['entity_count'].mean():.1f}")
print(f"  Max entity count: {df_ner['entity_count'].max()}")

In [ ]:
# --- Example: NER on 3 articles ---
print("Entity extraction examples:\n")
sample = df_ner.nlargest(3, 'entity_count')[['title', 'entities_org', 'entities_gpe', 'entities_person', 'entity_count']]
for _, row in sample.iterrows():
    print(f"TITLE  : {row['title'][:80]}")
    print(f"ORGs   : {row['entities_org'][:100]}")
    print(f"GPEs   : {row['entities_gpe'][:100]}")
    print(f"PERSONs: {row['entities_person'][:100]}")
    print(f"Count  : {row['entity_count']}")
    print()

In [ ]:
# Top organisations across the corpus
from collections import Counter

all_orgs = [org for orgs in df_ner['entities_org'].dropna() for org in orgs.split('|') if org.strip()]
org_counts = Counter(all_orgs).most_common(20)

fig, ax = plt.subplots(figsize=(9, 5))
orgs, counts = zip(*org_counts)
ax.barh(list(reversed(orgs)), list(reversed(counts)), color='#457B9D')
ax.set_xlabel('Mention count')
ax.set_title('Top 20 Organisations Mentioned in Threat Articles')
plt.tight_layout()
plt.savefig(OUTPUTS / 'top_organisations.png', bbox_inches='tight')
plt.show()

## 4 — Threat Classification

### Stage 1 — Keyword-based classifier
A lightweight, fully explainable approach: count how often known threat keywords appear
(normalised by article word count), assign to the highest-scoring category.

### Stage 2 — Zero-shot NLI (top-50 articles)
We apply a small cross-encoder NLI model (`nli-MiniLM2-L6-H768`, ~85 MB, CPU-only)
to the top-50 articles ranked by keyword score. This demonstrates ML-based classification
without requiring a labelled training set or GPU.

> **Design choice:** We run zero-shot only on the top-50 to keep runtime under 3 minutes.
> In a production system this would run on all articles with GPU acceleration.

In [ ]:
# Stage 1 only first (fast — no model download needed)
df_classified = classify_dataframe(df_ner, run_zero_shot=False)

print(f"\n✓ Keyword classification complete")
print(df_classified['threat_category'].value_counts())

In [ ]:
# Threat category distribution
cat_counts = df_classified['threat_category'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(cat_counts.index, cat_counts.values, color=PALETTE[:len(cat_counts)])
ax.set_ylabel('Number of articles')
ax.set_title('Article Distribution by Threat Category')
ax.set_xticklabels(cat_counts.index, rotation=20, ha='right')
for bar, val in zip(bars, cat_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val),
            ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUTS / 'threat_category_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# Stage 2: Zero-shot on top-50 (downloads ~85 MB model on first run)
# Comment out if offline or time-constrained
print("Running zero-shot classification on top-50 articles ...")
print("(First run downloads ~85 MB model — subsequent runs use cache)")

df_classified = classify_dataframe(df_ner, run_zero_shot=True)

# Agreement analysis between keyword and zero-shot
zs_mask = df_classified['zs_category'].notna()
agree = (df_classified.loc[zs_mask, 'threat_category'] == df_classified.loc[zs_mask, 'zs_category']).mean()
print(f"\nKeyword vs Zero-shot agreement on top-50: {agree:.1%}")

# Show disagreements
disagree = df_classified[zs_mask & (df_classified['threat_category'] != df_classified['zs_category'])]
print(f"Disagreements ({len(disagree)}):")
display(disagree[['title', 'threat_category', 'zs_category', 'zs_confidence']].head(5))

## 5 — Risk Scoring

The composite risk score is computed as:

$$
\text{Risk Score} = \Big(0.35 \cdot S_{\text{keyword}} + 0.25 \cdot S_{\text{sentiment}} + 0.20 \cdot S_{\text{recency}} + 0.20 \cdot S_{\text{entity}}\Big) \times M_{\text{category}} \times 10
$$

Where:
- $S_{\text{keyword}}$ — normalised keyword density score $\in [0,1]$
- $S_{\text{sentiment}}$ — VADER negativity: $(1 - \text{compound}) / 2 \in [0,1]$
- $S_{\text{recency}}$ — exponential decay from publication date (half-life = 30 days)
- $S_{\text{entity}}$ — named entity density $\in [0,1]$
- $M_{\text{category}}$ — category severity multiplier (Terrorism: 1.5, Cyber: 1.3, etc.)

Final labels: **Low** (0–3), **Medium** (3–6), **High** (6–8), **Critical** (8–10)

In [ ]:
df_scored = score_dataframe(df_classified)

print(f"\n✓ Risk scoring complete")
print(f"  Score range: {df_scored['risk_score'].min():.2f} – {df_scored['risk_score'].max():.2f}")
print(f"  Mean score : {df_scored['risk_score'].mean():.2f}")
print(f"\nSeverity label distribution:")
display(df_scored['severity_label'].value_counts().reindex(['Critical','High','Medium','Low'], fill_value=0))

## 6 — Visualisations

In [ ]:
# 6a — Risk score distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(df_scored['risk_score'], bins=30, color='#E63946', edgecolor='white')
axes[0].axvline(df_scored['risk_score'].mean(), color='black', linestyle='--', label=f"Mean: {df_scored['risk_score'].mean():.2f}")
axes[0].set_xlabel('Risk Score (0–10)')
axes[0].set_ylabel('Article count')
axes[0].set_title('Distribution of Risk Scores')
axes[0].legend()

# Severity pie
sev_counts = df_scored['severity_label'].value_counts().reindex(['Critical','High','Medium','Low'], fill_value=0)
sev_colors = ['#E63946', '#F4A261', '#2A9D8F', '#A8DADC']
axes[1].pie(sev_counts.values, labels=sev_counts.index, colors=sev_colors,
            autopct='%1.1f%%', startangle=90, pctdistance=0.8)
axes[1].set_title('Severity Label Distribution')

plt.tight_layout()
plt.savefig(OUTPUTS / 'risk_score_dist.png', bbox_inches='tight')
plt.show()

In [ ]:
# 6b — Score vs sentiment scatter (coloured by category)
fig, ax = plt.subplots(figsize=(9, 5))

categories = df_scored['threat_category'].unique()
color_map = {cat: col for cat, col in zip(categories, PALETTE)}

for cat in categories:
    subset = df_scored[df_scored['threat_category'] == cat]
    ax.scatter(
        subset['sentiment_neg'], subset['risk_score'],
        c=color_map[cat], label=cat, alpha=0.6, s=20
    )

ax.set_xlabel('Sentiment negativity score (0=positive, 1=very negative)')
ax.set_ylabel('Composite risk score (0–10)')
ax.set_title('Risk Score vs Sentiment Negativity by Threat Category')
ax.legend(loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUTS / 'risk_vs_sentiment.png', bbox_inches='tight')
plt.show()

In [ ]:
# 6c — Risk score by category (box plot)
fig, ax = plt.subplots(figsize=(9, 4))

order = df_scored.groupby('threat_category')['risk_score'].median().sort_values(ascending=False).index.tolist()
data_by_cat = [df_scored[df_scored['threat_category'] == cat]['risk_score'].values for cat in order]

bp = ax.boxplot(data_by_cat, patch_artist=True, labels=order)
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Risk Score')
ax.set_title('Risk Score Distribution by Threat Category')
ax.set_xticklabels(order, rotation=15, ha='right')
plt.tight_layout()
plt.savefig(OUTPUTS / 'risk_by_category.png', bbox_inches='tight')
plt.show()

In [ ]:
# 6d — Entity word cloud
all_entity_text = ' '.join(
    df_scored['entities_org'].fillna('') + ' ' +
    df_scored['entities_gpe'].fillna('') + ' ' +
    df_scored['entities_person'].fillna('')
).replace('|', ' ')

wc = WordCloud(
    width=900, height=400, background_color='white',
    colormap='Reds', max_words=120
).generate(all_entity_text)

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(wc, interpolation='bilinear')
ax.axis('off')
ax.set_title('Named Entities in Threat Articles (ORG + GPE + PERSON)', fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUTS / 'entity_wordcloud.png', bbox_inches='tight')
plt.show()

In [ ]:
# 6e — Articles over time by category
df_scored['month'] = df_scored['published'].dt.to_period('M')
time_cat = df_scored.groupby(['month', 'threat_category']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(11, 4))
time_cat.plot(ax=ax, colormap='tab10', marker='o', markersize=3)
ax.set_xlabel('Month')
ax.set_ylabel('Article count')
ax.set_title('Monthly Article Volume by Threat Category')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(OUTPUTS / 'articles_over_time.png', bbox_inches='tight')
plt.show()

## 7 — Threat Report

The threat report ranks all articles by composite risk score and exports
to `outputs/threat_report.csv` and `outputs/threat_report.json`.

In [ ]:
# Print top-10 threat summary
print_summary(df_scored, top_n=10)

In [ ]:
# Export full report
csv_path, json_path = export_report(df_scored)
print(f"\nFull report exported to:\n  {csv_path}\n  {json_path}")

# Display as styled table
report_df = generate_threat_report(df_scored).head(15)

def color_severity(val):
    colors = {'Critical': 'background-color:#FFCCCC', 'High': 'background-color:#FFE0B2',
               'Medium': 'background-color:#FFF9C4', 'Low': 'background-color:#E8F5E9'}
    return colors.get(val, '')

styled = (
    report_df[['title', 'published', 'threat_category', 'severity_label', 'risk_score', 'entities_org']]
    .style
    .applymap(color_severity, subset=['severity_label'])
    .format({'risk_score': '{:.2f}', 'published': lambda d: str(d)[:10] if pd.notna(d) else 'N/A'})
    .set_caption('Top 15 Threats by Composite Risk Score')
)
display(styled)

In [ ]:
# Intelligence Briefs — top 2 per category
briefs = intelligence_briefs(df_scored, top_per_category=2)

print("\n=== INTELLIGENCE BRIEFS ===")
for category, items in briefs.items():
    print(f"\n{'─'*60}")
    print(f"  [{category}]")
    for item in items:
        print(f"  Title    : {item['title'][:70]}")
        print(f"  Score    : {item['risk_score']:.2f}/10  ({item['severity']})")
        print(f"  Date     : {item['published']}")
        if item['key_orgs']:
            print(f"  Orgs     : {item['key_orgs'][:60]}")
        if item['key_locations']:
            print(f"  Locations: {item['key_locations'][:60]}")
        print(f"  URL      : {item['url']}")
        print()

## 8 — Discussion, Limitations & Future Work

### 8.1 Data Quality & Limitations

- **Source bias:** The Guardian is a single outlet with editorial perspective. A production system should aggregate multiple independent sources (Reuters, AP, BBC, Al Jazeera) to reduce bias.
- **Language limitation:** Only English-language articles are processed. Many significant threats originate from non-English sources (Russian, Chinese, Arabic).
- **Historical window:** We collect only the past 6 months. Slow-burn campaigns or long-tail threats may be under-represented.
- **Guardian developer tier:** Page size capped at 50; some fields may be truncated. A commercial API tier removes these restrictions.

### 8.2 AI/ML Approach Limitations

- **Keyword classifier:** Sensitive to vocabulary drift; novel attack terminology not in the taxonomy is missed.
- **Zero-shot classifier:** Confidences can be poorly calibrated; no domain-specific fine-tuning.
- **VADER sentiment:** Designed for social media; may mis-score formal journalistic prose.
- **Risk score weights:** Currently hand-tuned; a labelled ground-truth dataset would allow data-driven weight optimisation.

### 8.3 Ethical & Legal Considerations

- All articles are publicly available under The Guardian's open API terms.
- The system does not scrape social media, personal data, or restricted sources.
- Automated threat labelling could be misused; human-in-the-loop review is essential before acting on scores.
- False positives (legitimate articles scored as high-risk) could cause harm if acted upon without verification.

### 8.4 Scalability Path

| Component | Prototype | Production |
|-----------|-----------|------------|
| Storage | CSV files | PostgreSQL + vector DB (Weaviate/Pinecone) |
| Ingestion | Manual run | Scheduled pipeline (Apache Airflow / Prefect) |
| NLP | spaCy small | Transformer-based NER (GLiNER, BERT-NER) |
| Classification | Keyword + MiniLM | Fine-tuned cybersecurity BERT |
| UI | Notebook | Streamlit dashboard / REST API |
| Multi-source | Guardian only | CISA + CVE + social media + dark web OSINT |

### 8.5 Potential Future Improvements

1. **MITRE ATT&CK mapping** — extract TTPs and map extracted entities to ATT&CK techniques automatically
2. **Graph-based threat correlation** — build a knowledge graph linking actors, tools, and campaigns
3. **Temporal trend detection** — alert when a threat category shows sudden volume spike
4. **Multi-source fusion** — combine Guardian + CISA advisories + CVE feeds with cross-source deduplication
5. **Active learning** — analyst feedback loop to continuously improve classifier accuracy

In [ ]:
# Final summary statistics for the report
print("=" * 50)
print("PROTOTYPE SUMMARY STATISTICS")
print("=" * 50)
print(f"Total articles collected  : {len(df_scored)}")
print(f"Collection window         : {DATE_FROM} → {DATE_TO}")
print(f"Threat categories found   : {df_scored['threat_category'].nunique()}")
print(f"Mean risk score           : {df_scored['risk_score'].mean():.2f} / 10")
print(f"High/Critical articles    : {(df_scored['severity_label'].isin(['High','Critical'])).sum()} ({(df_scored['severity_label'].isin(['High','Critical'])).mean():.1%})")
print(f"Avg entities per article  : {df_scored['entity_count'].mean():.1f}")
print(f"Zero-shot sample size     : {df_scored['zs_category'].notna().sum()}")
print("=" * 50)